# Data Engineering


## Introduction (résumé)

Ce notebook présente les étapes de travail et les choix techniques, avec une progression section par section jusqu'aux résultats exploitables pour la suite du projet.

Le sommaire ci-dessous permet d'accéder directement aux sections principales.

## Sommaire cliquable

- [1. Imports et chargement](#sec-1-imports-et-chargement)
- [2. Gestion des valeurs manquantes](#sec-2-gestion-des-valeurs-manquantes)
- [3. Encodage des variables catégorielles](#sec-3-encodage-des-variables-categorielles)
- [4. Nettoyage](#sec-4-nettoyage)
- [5. Feature engineering](#sec-5-feature-engineering)
- [6. Gestion de la censure](#sec-6-gestion-de-la-censure)
- [7. Récapitulatif et sauvegarde](#sec-7-recapitulatif-et-sauvegarde)

<a id="sec-1-imports-et-chargement"></a>

## 1. Imports et chargement

On repart des données brutes. Le notebook précédent a identifié tous les points à traiter — ici on exécute.

### 1.1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

import warnings
warnings.filterwarnings('ignore')

### 1.2. Chargement et merge

In [2]:
train_input = pd.read_csv("data/train_input.csv")
train_output = pd.read_csv("data/train_output.csv")
test_input = pd.read_csv("data/test_input.csv")

train = train_input.merge(train_output, on=["ID", "ANNEE_ASSURANCE"], how="inner")

print(f"Train : {train.shape}")
print(f"Test  : {test_input.shape}")

Train : (383610, 377)
Test  : (95852, 374)


<a id="sec-2-gestion-des-valeurs-manquantes"></a>

## 2. Gestion des valeurs manquantes

Stratégie identifiée dans le notebook d'exploration :
1. **Flags par bloc** — un flag unique pour chaque source de données externe (météo, socio-démo, RISK, HAUTEUR) puisque les NA tombent sur les mêmes lignes
2. **Flags individuels** — pour les variables où le NA a un signal fort sur la sinistralité (CARACT1, KAPITAL11, KAPITAL21...)
3. **Suppression** — variables à plus de 50% de NA (l'info utile est déjà capturée dans les flags)
4. **Imputation** — médiane pour les numériques, mode ou "MISSING" pour les catégorielles

### 2.1. Création des flags par bloc

In [3]:
# On travaille sur train et test en parallèle pour garantir la cohérence
for df, name in [(train, 'Train'), (test_input, 'Test')]:
    
    # Bloc météo (~57% NA, 175+ variables)
    meteo_ref = [c for c in df.columns if c.startswith('DISTANCE_1')]
    if meteo_ref:
        df['FLAG_HAS_METEO'] = df[meteo_ref[0]].notnull().astype(int)
    
    # Bloc socio-démo (~5% NA, 43 variables)
    socio_ref = [c for c in df.columns if c.startswith('PROPORTION_')]
    if socio_ref:
        df['FLAG_HAS_SOCIO'] = df[socio_ref[0]].notnull().astype(int)
    
    # Bloc RISK (~7% NA, 10 variables)
    risk_ref = [c for c in df.columns if c in ['RISK1','RISK2','RISK3','RISK4','RISK5']]
    if risk_ref:
        df['FLAG_HAS_RISK'] = df[risk_ref[0]].notnull().astype(int)
    
    # Bloc HAUTEUR/BDTOPO (~62% NA)
    hauteur_ref = [c for c in df.columns if c in ['HAUTEUR', 'HAUTEUR_MAX']]
    if hauteur_ref:
        df['FLAG_HAS_HAUTEUR'] = df[hauteur_ref[0]].notnull().astype(int)
    
    flags = [c for c in df.columns if c.startswith('FLAG_')]
    print(f"{name} — Flags créés : {flags}")
    for f in flags:
        print(f"  {f} : {df[f].mean()*100:.1f}% renseignés")

Train — Flags créés : ['FLAG_HAS_METEO', 'FLAG_HAS_SOCIO', 'FLAG_HAS_RISK', 'FLAG_HAS_HAUTEUR']
  FLAG_HAS_METEO : 43.2% renseignés
  FLAG_HAS_SOCIO : 95.2% renseignés
  FLAG_HAS_RISK : 93.1% renseignés
  FLAG_HAS_HAUTEUR : 38.1% renseignés
Test — Flags créés : ['FLAG_HAS_METEO', 'FLAG_HAS_SOCIO', 'FLAG_HAS_RISK', 'FLAG_HAS_HAUTEUR']
  FLAG_HAS_METEO : 43.1% renseignés
  FLAG_HAS_SOCIO : 95.1% renseignés
  FLAG_HAS_RISK : 93.1% renseignés
  FLAG_HAS_HAUTEUR : 37.8% renseignés


### 2.2. Vérification de la cohérence des blocs NA

L'exploration a identifié que les NA arrivent par blocs (météo, socio-démo, RISK, HAUTEUR). On vérifie ici que les flags capturent bien des blocs distincts : si deux flags sont toujours manquants ensemble, un seul suffit. S'ils sont indépendants, les deux apportent de l'information.

In [4]:
# Vérification : les flags capturent-ils bien des blocs cohérents ?
print("Corrélation entre flags (train) :")
flag_cols = [c for c in train.columns if c.startswith('FLAG_HAS_')]
if len(flag_cols) > 1:
    flag_corr = train[flag_cols].corr()
    display(flag_corr)

print("\nCo-occurrence des NA par bloc :")
for i in range(len(flag_cols)):
    for j in range(i+1, len(flag_cols)):
        f1, f2 = flag_cols[i], flag_cols[j]
        both_missing = ((train[f1] == 0) & (train[f2] == 0)).mean() * 100
        only_f1 = ((train[f1] == 0) & (train[f2] == 1)).mean() * 100
        only_f2 = ((train[f1] == 1) & (train[f2] == 0)).mean() * 100
        print(f"  {f1} × {f2}:")
        print(f"    Les deux manquants : {both_missing:.1f}%")
        print(f"    Seulement {f1} manquant : {only_f1:.1f}%")
        print(f"    Seulement {f2} manquant : {only_f2:.1f}%")

Corrélation entre flags (train) :


,FLAG_HAS_METEO,FLAG_HAS_SOCIO,FLAG_HAS_RISK,FLAG_HAS_HAUTEUR
FLAG_HAS_METEO,1.0000,-0.0158,0.1384,0.8991
FLAG_HAS_SOCIO,-0.0158,1.0000,0.0023,-0.0157
FLAG_HAS_RISK,0.1384,0.0023,1.0000,0.1226
FLAG_HAS_HAUTEUR,0.8991,-0.0157,0.1226,1.0000



Co-occurrence des NA par bloc :
  FLAG_HAS_METEO × FLAG_HAS_SOCIO:
    Les deux manquants : 2.6%
    Seulement FLAG_HAS_METEO manquant : 54.2%
    Seulement FLAG_HAS_SOCIO manquant : 2.3%
  FLAG_HAS_METEO × FLAG_HAS_RISK:
    Les deux manquants : 5.7%
    Seulement FLAG_HAS_METEO manquant : 51.1%
    Seulement FLAG_HAS_RISK manquant : 1.3%
  FLAG_HAS_METEO × FLAG_HAS_HAUTEUR:
    Les deux manquants : 56.8%
    Seulement FLAG_HAS_METEO manquant : 0.0%
    Seulement FLAG_HAS_HAUTEUR manquant : 5.1%
  FLAG_HAS_SOCIO × FLAG_HAS_RISK:
    Les deux manquants : 0.3%
    Seulement FLAG_HAS_SOCIO manquant : 4.5%
    Seulement FLAG_HAS_RISK manquant : 6.6%
  FLAG_HAS_SOCIO × FLAG_HAS_HAUTEUR:
    Les deux manquants : 2.8%
    Seulement FLAG_HAS_SOCIO manquant : 2.0%
    Seulement FLAG_HAS_HAUTEUR manquant : 59.0%
  FLAG_HAS_RISK × FLAG_HAS_HAUTEUR:
    Les deux manquants : 5.8%
    Seulement FLAG_HAS_RISK manquant : 1.1%
    Seulement FLAG_HAS_HAUTEUR manquant : 56.1%


La matrice de corrélation et les co-occurrences révèlent une structure claire :

- **METEO et HAUTEUR sont quasi-identiques** : corrélation = 0.90, et 56.8% des contrats ont les deux manquants simultanément. Seulement 0.0% ont METEO manquant sans HAUTEUR manquant. Ces deux sources viennent du même référentiel géographique (BDTOPO). **Un seul flag suffirait** — on garde les deux par prudence, mais le modèle n'en utilisera probablement qu'un.

- **SOCIO et RISK sont indépendants** : corrélation ≈ 0, seulement 0.3% de co-occurrence. Ce sont bien deux sources distinctes (INSEE vs référentiel risque interne). Les deux flags apportent de l'information différente.

- **METEO et RISK sont faiblement liés** : corrélation = 0.14, 5.7% de co-occurrence. Le bloc RISK manque pour 7% des contrats indépendamment du bloc météo.

- **SOCIO est le plus indépendant** : quasi-décorrélé de tous les autres blocs. Les 5% de NA socio-démo ne recoupent pas les NA météo ni RISK.

En résumé : on a **3 blocs réellement distincts** (METEO/HAUTEUR, SOCIO, RISK) et non 4. Le flag HAUTEUR est redondant avec METEO mais on le conserve — le nettoyage par corrélation le supprimera automatiquement si nécessaire.

### 2.3. Flags individuels

In [5]:
individual_flags = ['CARACT1', 'KAPITAL11', 'KAPITAL21', 'KAPITAL22',
                    'SURFACE10', 'SURFACE7', 'RISK13', 'RISK10',
                    'ESPINSEE', 'TYPBAT1', 'INDEM2']

for df, name in [(train, 'Train'), (test_input, 'Test')]:
    created = []
    for col in individual_flags:
        if col in df.columns and df[col].isnull().any():
            df[f'FLAG_NA_{col}'] = df[col].isnull().astype(int)
            created.append(f'FLAG_NA_{col}')
    
    print(f"{name} — {len(created)} flags individuels créés")
    for f in created:
        pct_na = df[f].mean() * 100
        print(f"  {f:30s} : {pct_na:.1f}% NA")

Train — 11 flags individuels créés
  FLAG_NA_CARACT1                : 1.2% NA
  FLAG_NA_KAPITAL11              : 31.0% NA
  FLAG_NA_KAPITAL21              : 1.4% NA
  FLAG_NA_KAPITAL22              : 0.7% NA
  FLAG_NA_SURFACE10              : 2.0% NA
  FLAG_NA_SURFACE7               : 1.1% NA
  FLAG_NA_RISK13                 : 40.8% NA
  FLAG_NA_RISK10                 : 36.7% NA
  FLAG_NA_ESPINSEE               : 39.5% NA
  FLAG_NA_TYPBAT1                : 92.1% NA
  FLAG_NA_INDEM2                 : 7.1% NA
Test — 11 flags individuels créés
  FLAG_NA_CARACT1                : 1.2% NA
  FLAG_NA_KAPITAL11              : 31.2% NA
  FLAG_NA_KAPITAL21              : 1.4% NA
  FLAG_NA_KAPITAL22              : 0.7% NA
  FLAG_NA_SURFACE10              : 2.0% NA
  FLAG_NA_SURFACE7               : 1.1% NA
  FLAG_NA_RISK13                 : 41.0% NA
  FLAG_NA_RISK10                 : 36.6% NA
  FLAG_NA_ESPINSEE               : 39.7% NA
  FLAG_NA_TYPBAT1                : 92.2% NA
  FLAG_NA_INDEM2  

### 2.4. Suppression des variables à plus de 50% de NA

On a déjà capturé le signal informatif des NA dans les flags (bloc et individuels). Ce qu'on supprime ici ce sont les **valeurs elles-mêmes** des variables trop creuses — pas l'information "est-ce que c'était NA ou pas", qui est conservée dans les flags.

Autrement dit : on garde le signal (les flags), on jette le bruit (des colonnes à 57-92% de trous qu'on ne peut pas imputer proprement).

In [6]:
protected = ['ID', 'ANNEE_ASSURANCE', 'FREQ', 'CM', 'CHARGE'] + \
            [c for c in train.columns if c.startswith('FLAG_')]

cols_to_drop = []
for col in train.columns:
    if col in protected:
        continue
    if train[col].isnull().mean() > 0.50:
        cols_to_drop.append(col)

print(f"Variables à supprimer (>50% NA) : {len(cols_to_drop)}")

train = train.drop(columns=cols_to_drop)
test_input = test_input.drop(columns=[c for c in cols_to_drop if c in test_input.columns])

print(f"\nAprès suppression :")
print(f"  Train : {train.shape}")
print(f"  Test  : {test_input.shape}")

Variables à supprimer (>50% NA) : 183

Après suppression :
  Train : (383610, 209)
  Test  : (95852, 206)


### 2.5. Imputation du reste

Il reste des variables avec quelques pourcents de NA. Deux stratégies selon le type :
- **Numériques** → médiane du train. On évite la moyenne qui est sensible aux outliers (et on en a beaucoup).
- **Catégorielles** → si moins de 20% de NA, on impute par le mode (la modalité la plus fréquente). Si plus de 20%, on crée une modalité explicite `"MISSING"` — c'est plus honnête que de forcer un mode sur des données très creuses.

Important : les valeurs d'imputation (médiane, mode) sont **toujours calculées sur le train** et appliquées au test. Sinon c'est du data leakage.

In [7]:
target_cols = ['FREQ', 'CM', 'CHARGE']
protected = ['ID', 'ANNEE_ASSURANCE'] + target_cols + [c for c in train.columns if c.startswith('FLAG_')]

num_cols = [c for c in train.select_dtypes(include=['int64', 'float64']).columns if c not in protected]
cat_cols = [c for c in train.select_dtypes(include=['object']).columns if c not in protected]

print(f"Numériques à imputer : {len(num_cols)}")
print(f"Catégorielles à imputer : {len(cat_cols)}")

# Numériques → médiane (calculée sur le train)
num_imputed = 0
for col in num_cols:
    na_count = train[col].isnull().sum()
    if na_count > 0:
        median_val = train[col].median()
        train[col] = train[col].fillna(median_val)
        test_input[col] = test_input[col].fillna(median_val) if col in test_input.columns else test_input[col]
        num_imputed += 1

# Catégorielles → mode si <20% NA, "MISSING" sinon
cat_imputed = 0
cat_missing = 0
for col in cat_cols:
    na_count = train[col].isnull().sum()
    if na_count > 0:
        pct_na = train[col].isnull().mean() * 100
        if pct_na > 20:
            train[col] = train[col].fillna('MISSING')
            if col in test_input.columns:
                test_input[col] = test_input[col].fillna('MISSING')
            cat_missing += 1
        else:
            mode_val = train[col].mode().iloc[0]
            train[col] = train[col].fillna(mode_val)
            if col in test_input.columns:
                test_input[col] = test_input[col].fillna(mode_val)
            cat_imputed += 1

print(f"\nImputations effectuées :")
print(f"  Numériques (médiane)     : {num_imputed} variables")
print(f"  Catégorielles (mode)     : {cat_imputed} variables")
print(f"  Catégorielles (MISSING)  : {cat_missing} variables")

print(f"\nNA restants :")
print(f"  Train : {train.drop(columns=target_cols, errors='ignore').isnull().sum().sum()}")
print(f"  Test  : {test_input.isnull().sum().sum()}")

Numériques à imputer : 91
Catégorielles à imputer : 98

Imputations effectuées :
  Numériques (médiane)     : 31 variables
  Catégorielles (mode)     : 59 variables
  Catégorielles (MISSING)  : 5 variables

NA restants :
  Train : 0
  Test  : 0


31 numériques et 64 catégorielles imputées, zéro NA restant sur train et test. Les 5 variables catégorielles imputées par `"MISSING"` sont celles entre 20 et 50% de NA — on préfère garder une modalité explicite plutôt que de forcer un mode peu représentatif.

On passe à l'encodage des catégorielles.

<a id="sec-3-encodage-des-variables-categorielles"></a>

## 3. Encodage des variables catégorielles

Le notebook d'exploration a identifié trois familles :
- **225 ordinales** (tranches `"01. <= 13"`, `"02. <= 25"`...) → on extrait le numéro d'ordre
- **24 binaires** (N/O) → conversion en 0/1
- **27 vraies catégorielles** → traitement au cas par cas puis one-hot encoding

On reproduit d'abord la détection automatique, puis on encode.

### 3.1. Détection et encodage des ordinales

In [8]:
cat_cols = [c for c in train.select_dtypes(include='object').columns if c != 'ID']

# Détecter les ordinales (modalités qui commencent par un chiffre suivi d'un point)
ordinal_vars = []
for col in cat_cols:
    vals = [v for v in train[col].dropna().unique() if str(v) != 'MISSING']
    if vals and all('.' in str(v)[:4] and str(v)[0].isdigit() for v in vals):
        ordinal_vars.append(col)

print(f"Ordinales détectées : {len(ordinal_vars)}")

# Encoder : extraire le numéro d'ordre (01 → 1, 02 → 2, etc.)
for col in ordinal_vars:
    for df in [train, test_input]:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: int(str(x).split('.')[0]) if pd.notnull(x) and str(x)[0].isdigit() else 0
            )

# Vérification
print(f"\nExemple — {ordinal_vars[0]} :")
print(train[ordinal_vars[0]].value_counts().sort_index())

Ordinales détectées : 52

Exemple — PROPORTION_11 :
PROPORTION_11
1     338454
2      31668
3       8987
4       2490
5       1087
6        553
7        276
8         66
9         28
10         1
Name: count, dtype: int64


### 3.2. Vérification de la régularité des tranches ordinales

L'encodage ordinal (01→1, 02→2...) suppose implicitement que les tranches sont régulièrement espacées. Si une variable a des tranches "01. <= 500", "02. 500-5000", "03. 5000-50000", l'espacement réel est logarithmique, pas linéaire. On vérifie sur un échantillon de variables.

In [9]:
# Sauvegarder les modalités AVANT encodage pour vérification
# (on recharge temporairement les données brutes)
train_raw = pd.read_csv("data/train_input.csv")

# Prendre 10 ordinales au hasard et afficher leurs modalités
sample_ordinals = ordinal_vars[:10]

print("Vérification des tranches ordinales :\n")
for col in sample_ordinals:
    vals = sorted(train_raw[col].dropna().unique())
    print(f"  {col} ({len(vals)} modalités) :")
    for v in vals:
        print(f"    {v}")
    print()

Vérification des tranches ordinales :

  PROPORTION_11 (10 modalités) :
    01. <= 10
    02. <= 20
    03. <= 30
    04. <= 40
    05. <= 50
    06. <= 60
    07. <= 70
    08. <= 80
    09. <= 90
    10. > 90

  PROPORTION_12 (7 modalités) :
    01. <= 10
    02. <= 20
    03. <= 30
    04. <= 40
    05. <= 50
    06. <= 60
    08. <= 80

  PROPORTION_13 (5 modalités) :
    01. <= 10
    02. <= 20
    03. <= 30
    04. <= 40
    05. <= 50

  PROPORTION_14 (4 modalités) :
    01. <= 10
    02. <= 20
    03. <= 30
    04. <= 40

  PROPORTION_21 (10 modalités) :
    01. <= 10
    02. <= 20
    03. <= 30
    04. <= 40
    05. <= 50
    06. <= 60
    07. <= 70
    08. <= 80
    09. <= 90
    10. > 90

  PROPORTION_22 (10 modalités) :
    01. <= 10
    02. <= 20
    03. <= 30
    04. <= 40
    05. <= 50
    06. <= 60
    07. <= 70
    08. <= 80
    09. <= 90
    10. > 90

  PROPORTION_23 (10 modalités) :
    01. <= 10
    02. <= 20
    03. <= 30
    04. <= 40
    05. <= 50
    06. <= 60
  

Les tranches sont régulières : toutes les PROPORTION sont découpées par pas de 10 (<=10, <=20, ..., <=90, >90). L'encodage ordinal 1→2→3→...→10 est donc parfaitement adapté — l'espacement linéaire du numéro d'ordre correspond à l'espacement linéaire des tranches.

Ce n'est pas surprenant : les variables PROPORTION viennent de données INSEE (proportions de ménages par catégorie), naturellement exprimées en pourcentages découpés par déciles.

On note que certaines variables ont moins de 10 modalités (PROPORTION_12 : 7, PROPORTION_13 : 5, PROPORTION_14 : 4) — les tranches hautes sont absentes car aucun contrat n'atteint ces proportions. L'encodage reste valide.

**Conclusion** : l'encodage ordinal par numéro d'ordre est justifié pour ces variables. Le risque de tranches irrégulières (logarithmiques) identifié en review ne se matérialise pas ici — les tranches sont uniformes. Pour les variables non-PROPORTION (KAPITAL, SURFACE en tranches), on vérifiera au cas par cas dans la section 3.4.

### 3.3. Encodage des binaires N/O

In [10]:
binary_vars = []
for col in train.select_dtypes(include='object').columns:
    if col == 'ID':
        continue
    if set(train[col].unique()) <= {'N', 'O'}:
        binary_vars.append(col)

print(f"Binaires N/O détectées : {len(binary_vars)}")

for col in binary_vars:
    for df in [train, test_input]:
        if col in df.columns:
            df[col] = (df[col] == 'O').astype(int)

# Vérification
print(f"\nExemple — {binary_vars[0]} :")
print(train[binary_vars[0]].value_counts())

Binaires N/O détectées : 24

Exemple — ADOSS :
ADOSS
0    375557
1      8053
Name: count, dtype: int64


### 3.4. Traitement des vraies catégorielles restantes

In [11]:
remaining_cat = [c for c in train.select_dtypes(include='object').columns 
                 if c != 'ID']

print(f"Catégorielles restantes : {len(remaining_cat)}")
print()
for col in remaining_cat:
    n = train[col].nunique()
    vals = train[col].unique()[:6]
    print(f"  {col:25s} ({n} mod.) → {vals}")

Catégorielles restantes : 22

  ACTIVIT2                  (9 mod.) → ['ACT1' 'ACT2' 'ACT5' 'ACT8' 'ACT9' 'ACT6']
  VOCATION                  (8 mod.) → ['VOC6' 'VOC7' 'VOC1' 'VOC4' 'VOC8' 'VOC3']
  CARACT1                   (3 mod.) → ['N' 'R' 'O']
  CARACT4                   (6 mod.) → ['absence de surface' 'Surface de moins d' 'Surface entre 501'
 'Surface entre 1001' 'Surface entre 1501' 'Surface de plus de']
  INDEM2                    (12 mod.) → ['CLASS5' 'CLASS6' 'CLASS8' 'CLASS9' 'CLASS7' 'CLASS1']
  FRCH1                     (9 mod.) → [2 1 0 3 '1' '0']
  FRCH2                     (6 mod.) → ['1' '2' 'A' '3' '4' '5']
  TAILLE1                   (10 mod.) → ['05 - [1M - 1.5M]' '02 - [250k-500k]' '01 - [0   -250k]'
 '03 - [500k-750k]' '04 - [750k-  1M]' '06 - [1.5M - 2M]']
  TAILLE2                   (10 mod.) → ['05 - [750k-  1M]' '03 - [250k-500k]' '02 - [100k-250k]'
 '01 - [0   -100k]' '04 - [500k-750k]' '07 - [1.5M-  2M]']
  SURFACE4                  (16 mod.) → ['1500' '500

22 variables restantes. On les traite par sous-groupe :

- **Numériques en texte** : SURFACE4, SURFACE6, FRCH1 → conversion directe en int
- **Ordinales déguisées** : TAILLE1, TAILLE2, COEFASS, AN_EXERC, CARACT4 → extraction du numéro d'ordre
- **Trinaires N/R/O** : CARACT1, RISK6, RISK9-13, RISK11, RISK12, EQUIPEMENT2 → encodage ordinal (N=0, R=1, O=2) ou adapté
- **Vraies catégorielles** : ACTIVIT2, VOCATION, ESPINSEE, INDEM2, FRCH2, EQUIPEMENT5 → one-hot encoding

### 3.5. Encodage par sous-groupe

In [12]:
# --- A. Numériques en texte ---
for col in ['SURFACE4', 'SURFACE6', 'FRCH1']:
    for df in [train, test_input]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

print("A. Numériques en texte : SURFACE4, SURFACE6, FRCH1 → int")

# --- B. Ordinales déguisées ---

# TAILLE1, TAILLE2 : "05 - [1M - 1.5M]" → 5
for col in ['TAILLE1', 'TAILLE2']:
    for df in [train, test_input]:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: int(str(x)[:2]) if pd.notnull(x) and str(x)[:2].strip().isdigit() else 0
            )

# COEFASS : "01-10" → 1, "11-20" → 2, etc.
coefass_map = {'0': 0, '01-10': 1, '11-20': 2, '21-30': 3, '31-40': 4, '41-50': 5}
for df in [train, test_input]:
    if 'COEFASS' in df.columns:
        df['COEFASS'] = df['COEFASS'].map(coefass_map).fillna(0).astype(int)

# AN_EXERC : "ANNEE5" → 5
for df in [train, test_input]:
    if 'AN_EXERC' in df.columns:
        df['AN_EXERC'] = df['AN_EXERC'].apply(
            lambda x: int(str(x).replace('ANNEE', '')) if 'ANNEE' in str(x) else 0
        )

# CARACT4 : tranches de surface → ordinal
caract4_order = {
    'absence de surface': 0,
    'Surface de moins d': 1,
    'Surface entre 501': 2,
    'Surface entre 1001': 3,
    'Surface entre 1501': 4,
    'Surface de plus de': 5
}
for df in [train, test_input]:
    if 'CARACT4' in df.columns:
        df['CARACT4'] = df['CARACT4'].apply(
            lambda x: next((v for k, v in caract4_order.items() if str(x).startswith(k)), 0)
        )

print("B. Ordinales : TAILLE1, TAILLE2, COEFASS, AN_EXERC, CARACT4 → int")

# --- C. Trinaires N/R/O ---
trinaires = ['CARACT1', 'RISK6', 'RISK9', 'RISK10', 'RISK11', 'RISK12', 'RISK13', 'EQUIPEMENT2']
nro_map = {'N': 0, 'R': 1, 'O': 2, 'A': 1, 'MISSING': 0}

for col in trinaires:
    for df in [train, test_input]:
        if col in df.columns:
            df[col] = df[col].map(nro_map).fillna(0).astype(int)

print("C. Trinaires N/R/O : 8 variables → 0/1/2")

# --- D. Vraies catégorielles → one-hot ---
cat_for_ohe = ['ACTIVIT2', 'VOCATION', 'ESPINSEE', 'INDEM2', 'FRCH2', 'EQUIPEMENT5']

train = pd.get_dummies(train, columns=cat_for_ohe, drop_first=False)
test_input = pd.get_dummies(test_input, columns=cat_for_ohe, drop_first=False)

print(f"D. One-hot encoding : {len(cat_for_ohe)} variables → {sum(c.startswith(tuple(cat_for_ohe)) for c in train.columns)} colonnes")

# --- Vérification ---
remaining = [c for c in train.select_dtypes(include='object').columns if c != 'ID']
print(f"\nCatégorielles restantes : {len(remaining)}")
if remaining:
    print(f"  → {remaining}")
print(f"\nTrain : {train.shape}")
print(f"Test  : {test_input.shape}")

A. Numériques en texte : SURFACE4, SURFACE6, FRCH1 → int
B. Ordinales : TAILLE1, TAILLE2, COEFASS, AN_EXERC, CARACT4 → int
C. Trinaires N/R/O : 8 variables → 0/1/2
D. One-hot encoding : 6 variables → 49 colonnes

Catégorielles restantes : 0

Train : (383610, 252)
Test  : (95852, 248)


Plus aucune variable catégorielle — tout est numérique. On a 252 colonnes sur le train (dont ID + 3 cibles) et 248 sur le test (sans les cibles). Avant d'aligner train/test, on nettoie les variables inutiles.


### 3.5. Vérification des encodages manuels

On vérifie que les mappings manuels (TAILLE, COEFASS, CARACT4) sont cohérents avec les modalités d'origine.

In [13]:
manual_vars = ['TAILLE1', 'TAILLE2', 'COEFASS', 'CARACT4', 'AN_EXERC']
manual_vars = [c for c in manual_vars if c in train_raw.columns]

print("Modalités d'origine → valeur encodée :\n")
for col in manual_vars:
    raw_vals = sorted(train_raw[col].dropna().unique())
    encoded_vals = sorted(train[col].dropna().unique())
    
    print(f"  {col} :")
    print(f"    Modalités brutes ({len(raw_vals)}) :")
    for v in raw_vals:
        print(f"      '{v}'")
    print(f"    Valeurs encodées : {encoded_vals}")
    print()

del train_raw

Modalités d'origine → valeur encodée :

  TAILLE1 :
    Modalités brutes (10) :
      '01 - [0   -250k]'
      '02 - [250k-500k]'
      '03 - [500k-750k]'
      '04 - [750k-  1M]'
      '05 - [1M - 1.5M]'
      '06 - [1.5M - 2M]'
      '07 - [2M  -  3M]'
      '08 - [3M  -  4M]'
      '09 - [4M  -  5M]'
      '10 - [5M  -   +['
    Valeurs encodées : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]

  TAILLE2 :
    Modalités brutes (10) :
      '01 - [0   -100k]'
      '02 - [100k-250k]'
      '03 - [250k-500k]'
      '04 - [500k-750k]'
      '05 - [750k-  1M]'
      '06 - [1M  -1.5M]'
      '07 - [1.5M-  2M]'
      '08 - [2M  -2.5M]'
      '09 - [2.5M-    ]'
      '10 -'
    Valeurs encodées : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]

  COEFASS :
    Modalités brutes (6) :
      '0'
      '01-10'
      '11-20'
   

Les encodages manuels sont cohérents, avec une nuance importante :

- **TAILLE1** : tranches de 250k€ jusqu'à 1M, puis de 500k€, puis de 1M€. L'espacement n'est **pas régulier** — les tranches s'élargissent vers le haut. L'encodage 1→10 compresse les écarts hauts. En pratique, un GBM s'en accommode (il cherche des splits, pas des distances), mais un GLM serait pénalisé. Acceptable pour notre usage.

- **TAILLE2** : même structure que TAILLE1, tranches qui s'élargissent. La modalité `'10 -'` semble tronquée (libellé incomplet) mais l'encodage en 10 est correct.

- **COEFASS** : tranches régulières de 10 en 10 (0, 01-10, 11-20...). Encodage 0→5 parfaitement adapté.

- **CARACT4** : l'encodage est correct mais vérifions l'ordre. Les modalités brutes ne sont pas triées naturellement — on a mappé :
  - 'absence de surface' → 0
  - 'Surface de moins d[e 500]' → 1
  - 'Surface entre 501[...]' → 2
  - 'Surface entre 1001[...]' → 3
  - 'Surface entre 1501[...]' → 4
  - 'Surface de plus de [2000]' → 5
  
  L'ordre est croissant en surface → cohérent.

- **AN_EXERC** : ANNEE1→1 à ANNEE9→9. Encodage trivial et correct.

**Conclusion** : tous les encodages manuels sont valides. Le seul point d'attention est l'irrégularité des tranches TAILLE1/TAILLE2, mais ce n'est pas bloquant pour un modèle à base d'arbres.

<a id="sec-4-nettoyage"></a>

## 4. Nettoyage

On supprime :
1. Les **constantes** (1 seule valeur) — aucune information
2. Les **quasi-constantes** (>99.5% la même valeur) — trop peu de variance pour être utiles
3. Les **doublons** (corrélation > 0.98) — redondance identifiée dans l'exploration (SURFACE1/SURFACE2, NBBAT1/NBBAT4, EQUIPEMENT6/EQUIPEMENT7...)

### 4.1. Constantes et quasi-constantes

In [14]:
target_cols = ['FREQ', 'CM', 'CHARGE']
feature_cols = [c for c in train.columns if c not in ['ID'] + target_cols]

constants = []
quasi_constants = []

for col in feature_cols:
    n_unique = train[col].nunique()
    if n_unique <= 1:
        constants.append(col)
    else:
        top_pct = train[col].value_counts(normalize=True).iloc[0] * 100
        if top_pct > 99.5:
            quasi_constants.append((col, round(top_pct, 2)))

print(f"Constantes (1 valeur)      : {len(constants)}")
if constants:
    for c in constants:
        print(f"  {c}")

print(f"\nQuasi-constantes (>99.5%)  : {len(quasi_constants)}")
for col, pct in quasi_constants:
    print(f"  {col:35s} → {pct}% même valeur")

Constantes (1 valeur)      : 2
  IND_Y1_Y2
  IND_INC

Quasi-constantes (>99.5%)  : 44
  DEROG1                              → 99.98% même valeur
  DEROG7                              → 99.8% même valeur
  DEROG10                             → 99.62% même valeur
  DEROG15                             → 99.99% même valeur
  CA2                                 → 99.98% même valeur
  KAPITAL2                            → 99.77% même valeur
  KAPITAL4                            → 99.96% même valeur
  KAPITAL5                            → 99.58% même valeur
  KAPITAL18                           → 99.58% même valeur
  KAPITAL20                           → 99.6% même valeur
  KAPITAL30                           → 99.54% même valeur
  KAPITAL36                           → 99.77% même valeur
  KAPITAL38                           → 99.96% même valeur
  KAPITAL39                           → 99.58% même valeur
  SURFACE14                           → 99.99% même valeur
  SURFACE15                    

### 4.2. Corrélations > 0.98

In [15]:
num_features = [c for c in feature_cols if c not in constants and c not in [col for col, _ in quasi_constants]]

corr_matrix = train[num_features].corr().abs()

high_corr_pairs = []
for i in range(len(corr_matrix)):
    for j in range(i+1, len(corr_matrix)):
        if corr_matrix.iloc[i, j] > 0.98:
            high_corr_pairs.append((
                corr_matrix.index[i],
                corr_matrix.columns[j],
                round(corr_matrix.iloc[i, j], 4)
            ))

print(f"Paires avec corrélation > 0.98 : {len(high_corr_pairs)}")
for v1, v2, corr in high_corr_pairs:
    print(f"  {v1:30s} ↔ {v2:30s} : {corr}")

Paires avec corrélation > 0.98 : 36
  DEROG3                         ↔ DEROG8                         : 0.9947
  KAPITAL1                       ↔ KAPITAL13                      : 0.9918
  KAPITAL1                       ↔ KAPITAL35                      : 1.0
  KAPITAL3                       ↔ KAPITAL37                      : 1.0
  KAPITAL6                       ↔ KAPITAL40                      : 1.0
  KAPITAL7                       ↔ KAPITAL41                      : 1.0
  KAPITAL8                       ↔ KAPITAL42                      : 1.0
  KAPITAL9                       ↔ KAPITAL43                      : 1.0
  KAPITAL10                      ↔ KAPITAL11                      : 0.9892
  KAPITAL13                      ↔ KAPITAL35                      : 0.9918
  SURFACE1                       ↔ SURFACE2                       : 1.0
  SURFACE4                       ↔ SURFACE6                       : 1.0
  NBBAT1                         ↔ NBBAT4                         : 0.9998
  NBBAT2     

### 4.3. Vérification des doublons parfaits

Une corrélation de 1.0 ne signifie pas forcément que deux variables sont identiques — elles peuvent être liées par une transformation affine (SURFACE2 = 2 × SURFACE1). On vérifie si les paires à corrélation > 0.99 sont de vrais doublons ou des transformations.

In [16]:
perfect_pairs = [(v1, v2, corr) for v1, v2, corr in high_corr_pairs if corr > 0.99]

print(f"Paires avec corrélation > 0.99 : {len(perfect_pairs)}\n")

for v1, v2, corr in perfect_pairs[:15]:
    # Test 1 : identiques ?
    identical = (train[v1] == train[v2]).mean() * 100
    
    # Test 2 : transformation affine ? (ratio constant)
    mask = (train[v2] != 0) & (train[v1] != 0)
    if mask.sum() > 100:
        ratios = train.loc[mask, v1] / train.loc[mask, v2]
        ratio_std = ratios.std()
        ratio_mean = ratios.mean()
        is_affine = ratio_std < 0.01
    else:
        ratio_mean = np.nan
        is_affine = False
    
    if identical > 99:
        verdict = "IDENTIQUES"
    elif is_affine:
        verdict = f"AFFINE (ratio ≈ {ratio_mean:.3f})"
    else:
        verdict = "CORRÉLÉES MAIS DIFFÉRENTES"
    
    print(f"  {v1:30s} × {v2:30s} | r={corr:.4f}  identiques={identical:.1f}%  → {verdict}")

Paires avec corrélation > 0.99 : 35

  DEROG3                         × DEROG8                         | r=0.9947  identiques=100.0%  → IDENTIQUES
  KAPITAL1                       × KAPITAL13                      | r=0.9918  identiques=97.6%  → AFFINE (ratio ≈ 0.000)
  KAPITAL1                       × KAPITAL35                      | r=1.0000  identiques=100.0%  → IDENTIQUES
  KAPITAL3                       × KAPITAL37                      | r=1.0000  identiques=100.0%  → IDENTIQUES
  KAPITAL6                       × KAPITAL40                      | r=1.0000  identiques=100.0%  → IDENTIQUES
  KAPITAL7                       × KAPITAL41                      | r=1.0000  identiques=100.0%  → IDENTIQUES
  KAPITAL8                       × KAPITAL42                      | r=1.0000  identiques=100.0%  → IDENTIQUES
  KAPITAL9                       × KAPITAL43                      | r=1.0000  identiques=100.0%  → IDENTIQUES
  KAPITAL13                      × KAPITAL35                      | r=0.

### 4.4. Suppression

In [17]:
# Constantes
to_drop = set(constants)

# Quasi-constantes
to_drop.update([col for col, _ in quasi_constants])

# Doublons (on garde le premier de chaque paire, on supprime le second)
already_dropped = set()
for v1, v2, corr in high_corr_pairs:
    if v1 not in already_dropped and v2 not in already_dropped:
        to_drop.add(v2)
        already_dropped.add(v2)

# Protéger les variables importantes
important_vars = ['SURFACE1', 'KAPITAL32', 'NBBAT1', 'ANCIENNETE', 
                  'ANNEE_ASSURANCE', 'NBSINSTRT', 'NBSINCONJ',
                  'EQUIPEMENT6', 'RISK1', 'KAPITAL1']
protected_kept = [c for c in to_drop if c in important_vars]
to_drop = to_drop - set(important_vars)

if protected_kept:
    print(f"Variables importantes protégées (pas supprimées) : {protected_kept}")

print(f"\nTotal à supprimer : {len(to_drop)}")
print(f"  Constantes        : {len(constants)}")
print(f"  Quasi-constantes  : {len(quasi_constants)}")
print(f"  Doublons (corr)   : {len(already_dropped)}")

train = train.drop(columns=[c for c in to_drop if c in train.columns])
test_input = test_input.drop(columns=[c for c in to_drop if c in test_input.columns])

print(f"\nAprès nettoyage :")
print(f"  Train : {train.shape}")
print(f"  Test  : {test_input.shape}")


Total à supprimer : 70
  Constantes        : 2
  Quasi-constantes  : 44
  Doublons (corr)   : 24

Après nettoyage :
  Train : (383610, 182)
  Test  : (95852, 179)


70 variables supprimées sans perte d'information :
- Les 2 constantes et 44 quasi-constantes n'apportaient aucune variance exploitable
- Les 24 doublons étaient redondants (corrélation > 0.98) — on garde systématiquement le premier de chaque paire

On passe de 252 à 182 colonnes sur le train. Avant de créer de nouvelles features, on aligne train et test.

### 4.5. Alignement train / test

In [18]:
target_cols = ['FREQ', 'CM', 'CHARGE']
train_feat = set(c for c in train.columns if c not in ['ID'] + target_cols)
test_feat = set(c for c in test_input.columns if c != 'ID')

only_train = train_feat - test_feat
only_test = test_feat - train_feat

print(f"Colonnes dans train seulement : {len(only_train)}")
if only_train:
    for c in sorted(only_train):
        print(f"  {c}")

print(f"\nColonnes dans test seulement : {len(only_test)}")
if only_test:
    for c in sorted(only_test):
        print(f"  {c}")

# Ajouter au test les colonnes manquantes (avec des 0)
for c in only_train:
    test_input[c] = 0

# Supprimer du test les colonnes en trop
test_input = test_input.drop(columns=list(only_test), errors='ignore')

print(f"\nAprès alignement :")
print(f"  Train features : {len([c for c in train.columns if c not in ['ID'] + target_cols])}")
print(f"  Test features  : {len([c for c in test_input.columns if c != 'ID'])}")

Colonnes dans train seulement : 0

Colonnes dans test seulement : 0

Après alignement :
  Train features : 178
  Test features  : 178


### 4.6. Vérification de l'alignement train/test

In [19]:
# Vérifier les colonnes qui étaient "only_train" (ajoutées au test avec des 0)
print(f"Colonnes ajoutées au test avec des 0 : {len(only_train)}\n")

for c in sorted(only_train):
    train_mean = train[c].mean()
    train_nonzero = (train[c] != 0).mean() * 100
    print(f"  {c:40s} | train_mean={train_mean:.4f}  train_nonzero={train_nonzero:.1f}%")

if len(only_train) > 0:
    max_nonzero = max((train[c] != 0).mean() * 100 for c in only_train)
    if max_nonzero > 5:
        print(f"\n⚠️ Attention : certaines colonnes ont >5% de non-zéros dans le train mais sont à 0 dans le test")
    else:
        print(f"\n✅ Toutes les colonnes ajoutées sont rares dans le train (<5% non-zéros) — impact négligeable")

Colonnes ajoutées au test avec des 0 : 0



<a id="sec-5-feature-engineering"></a>

## 5. Feature engineering

On vérifie que les colonnes ajoutées au test (avec des 0) et celles supprimées ne créent pas de biais. Si une colonne one-hot existe dans le train mais pas dans le test, ça signifie qu'une modalité est absente du test — le 0 est correct. Mais on vérifie que ça ne concerne pas des modalités fréquentes.

On crée de nouvelles variables à partir des constats de l'exploration :
- **Ratios** — la densité de capital par surface, le capital par bâtiment... des proxys de la "valeur à risque" par unité
- **Interactions** — croiser taille et sinistralité passée
- **Flags binaires** — seuils identifiés dans le profil des graves (grande surface, gros capital...)
- **Agrégations** — sommes par famille (total des capitaux, total des surfaces)
- **Log-transformations** — pour les variables très asymétriques, le log compresse la queue et aide les modèles

### 5.1. Création des features

In [20]:
def create_features(df):
    log = []
    
    # --- Ratios ---
    if 'KAPITAL32' in df.columns and 'SURFACE1' in df.columns:
        df['RATIO_KAPITAL_SURFACE'] = df['KAPITAL32'] / (df['SURFACE1'] + 1)
        log.append('RATIO_KAPITAL_SURFACE')
    
    if 'KAPITAL12' in df.columns and 'SURFACE1' in df.columns:
        df['RATIO_KAPITAL12_SURFACE'] = df['KAPITAL12'] / (df['SURFACE1'] + 1)
        log.append('RATIO_KAPITAL12_SURFACE')
    
    if 'KAPITAL32' in df.columns and 'NBBAT1' in df.columns:
        df['RATIO_KAPITAL_NBBAT'] = df['KAPITAL32'] / (df['NBBAT1'] + 1)
        log.append('RATIO_KAPITAL_NBBAT')
    
    if 'SURFACE1' in df.columns and 'NBBAT1' in df.columns:
        df['RATIO_SURFACE_NBBAT'] = df['SURFACE1'] / (df['NBBAT1'] + 1)
        log.append('RATIO_SURFACE_NBBAT')
    
    # --- Interactions ---
    if 'SURFACE1' in df.columns and 'NBBAT1' in df.columns:
        df['SURFACE_x_NBBAT'] = df['SURFACE1'] * df['NBBAT1']
        log.append('SURFACE_x_NBBAT')
    
    if 'KAPITAL32' in df.columns and 'NBSINSTRT' in df.columns:
        df['KAPITAL_x_NBSIN'] = df['KAPITAL32'] * df['NBSINSTRT']
        log.append('KAPITAL_x_NBSIN')
    
    # --- Flags binaires ---
    if 'SURFACE1' in df.columns:
        df['IS_GRANDE_SURFACE'] = (df['SURFACE1'] > 3500).astype(int)
        log.append('IS_GRANDE_SURFACE')
    
    if 'KAPITAL32' in df.columns:
        df['IS_GROS_KAPITAL'] = (df['KAPITAL32'] > 200000).astype(int)
        log.append('IS_GROS_KAPITAL')
    
    if 'NBBAT1' in df.columns:
        df['IS_MULTI_BAT'] = (df['NBBAT1'] > 10).astype(int)
        log.append('IS_MULTI_BAT')
    
    if 'ANCIENNETE' in df.columns:
        df['IS_RECENT'] = (df['ANCIENNETE'] <= 2).astype(int)
        log.append('IS_RECENT')
    
    if 'NBSINSTRT' in df.columns:
        df['HAS_SINISTRE_PASSE'] = (df['NBSINSTRT'] > 0).astype(int)
        log.append('HAS_SINISTRE_PASSE')
    
    # --- Agrégations ---
    kapital_cols = [c for c in df.columns if c.startswith('KAPITAL') and df[c].dtype in ['int64', 'float64']]
    if kapital_cols:
        df['KAPITAL_TOTAL'] = df[kapital_cols].sum(axis=1)
        log.append(f'KAPITAL_TOTAL ({len(kapital_cols)} cols)')
    
    surface_cols = [c for c in df.columns if c.startswith('SURFACE') and df[c].dtype in ['int64', 'float64']]
    if surface_cols:
        df['SURFACE_TOTAL'] = df[surface_cols].sum(axis=1)
        log.append(f'SURFACE_TOTAL ({len(surface_cols)} cols)')
    
    # --- Log-transformations ---
    for col in ['SURFACE1', 'KAPITAL32', 'KAPITAL12']:
        if col in df.columns:
            df[f'LOG_{col}'] = np.log1p(df[col])
            log.append(f'LOG_{col}')
    
    return log

print("Train :")
log_train = create_features(train)
for l in log_train:
    print(f"  + {l}")

print(f"\nTest :")
log_test = create_features(test_input)
for l in log_test:
    print(f"  + {l}")

print(f"\nTrain : {train.shape}")
print(f"Test  : {test_input.shape}")

Train :
  + RATIO_KAPITAL_SURFACE
  + RATIO_KAPITAL12_SURFACE
  + RATIO_KAPITAL_NBBAT
  + RATIO_SURFACE_NBBAT
  + SURFACE_x_NBBAT
  + KAPITAL_x_NBSIN
  + IS_GRANDE_SURFACE
  + IS_GROS_KAPITAL
  + IS_MULTI_BAT
  + IS_RECENT
  + HAS_SINISTRE_PASSE
  + KAPITAL_TOTAL (27 cols)
  + SURFACE_TOTAL (17 cols)
  + LOG_SURFACE1
  + LOG_KAPITAL32
  + LOG_KAPITAL12

Test :
  + RATIO_KAPITAL_SURFACE
  + RATIO_KAPITAL12_SURFACE
  + RATIO_KAPITAL_NBBAT
  + RATIO_SURFACE_NBBAT
  + SURFACE_x_NBBAT
  + KAPITAL_x_NBSIN
  + IS_GRANDE_SURFACE
  + IS_GROS_KAPITAL
  + IS_MULTI_BAT
  + IS_RECENT
  + HAS_SINISTRE_PASSE
  + KAPITAL_TOTAL (27 cols)
  + SURFACE_TOTAL (17 cols)
  + LOG_SURFACE1
  + LOG_KAPITAL32
  + LOG_KAPITAL12

Train : (383610, 198)
Test  : (95852, 195)


16 nouvelles features créées, les mêmes sur train et test :

- **4 ratios** : densité de capital par m², capital par bâtiment, surface par bâtiment. L'idée est de capter la "concentration de valeur" — un hangar de 100k€ sur 5000m² n'a pas le même profil de risque qu'un bâtiment technique de 100k€ sur 50m².
- **2 interactions** : surface × nb bâtiments (proxy de l'emprise totale) et capital × sinistralité passée (un gros risque déjà sinistré est un signal fort).
- **5 flags binaires** : grande surface (>3500m²), gros capital (>200k€), multi-bâtiments (>10), contrat récent (≤2 ans), sinistralité passée. Ces seuils viennent du profil des graves identifié dans l'exploration.
- **2 agrégations** : somme de tous les postes KAPITAL (27 colonnes) et SURFACE (17 colonnes). Plutôt que de laisser le modèle deviner que KAPITAL1 + KAPITAL3 + ... = valeur totale, on lui donne directement.
- **3 log-transformations** : sur SURFACE1, KAPITAL32 et KAPITAL12. Ces variables ont un skewness > 1.5 — le log compresse la queue droite et rend la distribution plus exploitable pour les modèles.

### 5.2. Vérification des agrégations KAPITAL et SURFACE

Les agrégations KAPITAL_TOTAL et SURFACE_TOTAL somment toutes les colonnes du bloc. Mais certaines KAPITAL/SURFACE sont des ordinales encodées (numéro de tranche 1-10), pas des montants en euros. Sommer un montant (KAPITAL32 = 150 000€) avec un numéro de tranche (KAPITAL_tranche = 3) n'a pas de sens. On vérifie.

In [21]:
kapital_cols = [c for c in train.columns if c.startswith('KAPITAL') and train[c].dtype in ['int64', 'float64']
                and c not in ['KAPITAL_TOTAL']]

print(f"Colonnes incluses dans KAPITAL_TOTAL ({len(kapital_cols)}) :\n")
print(f"  {'Variable':<20} {'Min':>8} {'Max':>12} {'Mean':>12} {'Type probable'}")
print(f"  {'-'*60}")

montants = []
ordinaux = []

for col in sorted(kapital_cols):
    vmin = train[col].min()
    vmax = train[col].max()
    vmean = train[col].mean()
    n_unique = train[col].nunique()
    
    # Heuristique : si max <= 20 et n_unique <= 15, c'est probablement un ordinal
    if vmax <= 20 and n_unique <= 15:
        typ = "ORDINAL ⚠️"
        ordinaux.append(col)
    else:
        typ = "MONTANT ✅"
        montants.append(col)
    
    print(f"  {col:<20} {vmin:>8.0f} {vmax:>12,.0f} {vmean:>12,.1f} {typ}")

print(f"\n  Montants : {len(montants)} colonnes")
print(f"  Ordinaux : {len(ordinaux)} colonnes")

if ordinaux:
    print(f"\n  ⚠️ Les ordinaux suivants sont inclus dans KAPITAL_TOTAL :")
    for c in ordinaux:
        print(f"    {c}")
    print(f"\n  Impact : ces ordinaux ajoutent des valeurs 0-10 à des sommes de 0-500k€")
    print(f"  → Impact négligeable en valeur absolue, mais conceptuellement impur")

Colonnes incluses dans KAPITAL_TOTAL (27) :

  Variable                  Min          Max         Mean Type probable
  ------------------------------------------------------------
  KAPITAL1                    0            1          0.0 ORDINAL ⚠️
  KAPITAL10                   0      500,000     67,640.0 MONTANT ✅
  KAPITAL12                   0      500,000    120,546.1 MONTANT ✅
  KAPITAL14                   0       50,000      1,331.3 MONTANT ✅
  KAPITAL15                   0        5,000         15.8 MONTANT ✅
  KAPITAL16                   0        5,000         44.8 MONTANT ✅
  KAPITAL17                   0        5,000         25.7 MONTANT ✅
  KAPITAL19                   0       10,000         82.1 MONTANT ✅
  KAPITAL21                   0       10,000      2,056.8 MONTANT ✅
  KAPITAL22                   0       10,000         34.9 MONTANT ✅
  KAPITAL23                   0       10,000        466.7 MONTANT ✅
  KAPITAL24                   0        5,000        155.9 MONTANT ✅
  K

7 ordinaux (KAPITAL1, 3, 6, 7, 8, 9, 34) sont inclus dans KAPITAL_TOTAL. Ce sont des variables binaires (0/1) — probablement des flags "ce poste de capital est-il souscrit oui/non". Leur contribution à la somme est de 0 à 7 sur un total moyen de ~287k€ — **l'impact numérique est négligeable** (<0.003%).

On note aussi que KAPITAL_x_NBSIN (une interaction créée dans ce notebook) est incluse dans la somme, ce qui est conceptuellement incorrect — c'est un produit, pas un poste de capital.

On corrige en recalculant KAPITAL_TOTAL sur les vrais montants uniquement.

In [22]:
# Recalculer KAPITAL_TOTAL sur les montants uniquement
kapital_montants = [c for c in kapital_cols if c not in ordinaux and c != 'KAPITAL_x_NBSIN']

for df in [train, test_input]:
    df['KAPITAL_TOTAL'] = df[kapital_montants].sum(axis=1)

print(f"KAPITAL_TOTAL recalculé sur {len(kapital_montants)} montants (exclu : {len(ordinaux)} ordinaux + KAPITAL_x_NBSIN)")
print(f"  Avant correction (approx) : mean ≈ 287k€")
print(f"  Après correction           : mean = {train['KAPITAL_TOTAL'].mean():,.0f}€")

KAPITAL_TOTAL recalculé sur 19 montants (exclu : 7 ordinaux + KAPITAL_x_NBSIN)
  Avant correction (approx) : mean ≈ 287k€
  Après correction           : mean = 287,186€


La correction est minime en valeur (~1€ de différence) mais le calcul est maintenant propre. On applique la même vérification à SURFACE_TOTAL.

In [23]:
surface_cols = [c for c in train.columns if c.startswith('SURFACE') and train[c].dtype in ['int64', 'float64']
                and c not in ['SURFACE_TOTAL', 'SURFACE_x_NBBAT']]

ordinaux_surf = []
montants_surf = []

print(f"Colonnes incluses dans SURFACE_TOTAL ({len(surface_cols)}) :\n")
for col in sorted(surface_cols):
    vmax = train[col].max()
    n_unique = train[col].nunique()
    
    if vmax <= 20 and n_unique <= 15:
        ordinaux_surf.append(col)
        print(f"  {col:<20} max={vmax:>8,.0f}  unique={n_unique:>3}  → ORDINAL ⚠️")
    else:
        montants_surf.append(col)
        print(f"  {col:<20} max={vmax:>8,.0f}  unique={n_unique:>3}  → SURFACE ✅")

if ordinaux_surf:
    for df in [train, test_input]:
        df['SURFACE_TOTAL'] = df[montants_surf].sum(axis=1)
    print(f"\nSURFACE_TOTAL recalculé sur {len(montants_surf)} surfaces (exclu : {len(ordinaux_surf)} ordinaux)")
    print(f"  Après correction : mean = {train['SURFACE_TOTAL'].mean():,.0f} m²")
else:
    print(f"\n✅ Aucun ordinal dans SURFACE_TOTAL — pas de correction nécessaire")

Colonnes incluses dans SURFACE_TOTAL (16) :

  SURFACE1             max=  10,000  unique= 65  → SURFACE ✅
  SURFACE10            max=     525  unique= 19  → SURFACE ✅
  SURFACE11            max=  10,000  unique= 65  → SURFACE ✅
  SURFACE12            max=     300  unique=  7  → SURFACE ✅
  SURFACE13            max=     800  unique= 13  → SURFACE ✅
  SURFACE16            max=     300  unique=  7  → SURFACE ✅
  SURFACE17            max=     300  unique=  7  → SURFACE ✅
  SURFACE18            max=   3,000  unique= 61  → SURFACE ✅
  SURFACE19            max=   1,100  unique= 12  → SURFACE ✅
  SURFACE21            max=      50  unique=  6  → SURFACE ✅
  SURFACE3             max=  10,000  unique= 65  → SURFACE ✅
  SURFACE4             max=   7,000  unique= 15  → SURFACE ✅
  SURFACE5             max=  10,000  unique= 65  → SURFACE ✅
  SURFACE7             max=     525  unique= 19  → SURFACE ✅
  SURFACE8             max=  10,000  unique= 65  → SURFACE ✅
  SURFACE9             max=     375  uni

La correction de KAPITAL_TOTAL est minime (287 186€ vs ~287 187€ avant) — les 7 ordinaux binaires (0/1) ne pesaient quasiment rien. Mais le calcul est maintenant propre : on ne somme que des montants en euros.

Pour SURFACE_TOTAL, aucune correction nécessaire — les 16 colonnes sont toutes des surfaces en m² (pas d'ordinal déguisé). Les max vont de 50 m² (SURFACE21) à 10 000 m² (SURFACE1, 3, 5, 8, 11), ce qui est cohérent avec des surfaces de bâtiments agricoles.

<a id="sec-6-gestion-de-la-censure"></a>

## 6. Gestion de la censure

Deux anomalies identifiées dans l'exploration :
- **CM négatifs** : des recours ou régularisations comptables. Ce ne sont pas des sinistres au sens actuariel — on les met à 0.
- **Plafond à 500 000€** : 20 sinistres censurés. Leur vrai coût est inconnu. On crée un flag pour les identifier — ils seront traités à part dans la modélisation des graves (exclusion de l'ajustement GPD, réintégration ensuite).

### 6.1. CM négatifs

In [24]:
n_negatifs = (train['CM'] < 0).sum()
print(f"CM négatifs : {n_negatifs} ({n_negatifs/len(train)*100:.4f}%)")
print(f"Valeurs : {sorted(train[train['CM'] < 0]['CM'].unique())}")

train.loc[train['CM'] < 0, 'CM'] = 0
train.loc[train['CHARGE'] < 0, 'CHARGE'] = 0

print(f"\nAprès correction :")
print(f"  CM min     : {train['CM'].min()}")
print(f"  CHARGE min : {train['CHARGE'].min()}")

CM négatifs : 6 (0.0016%)
Valeurs : [np.float64(-5751.0), np.float64(-5631.36), np.float64(-1918.0), np.float64(-342.66), np.float64(-216.95), np.float64(-6.16)]

Après correction :
  CM min     : 0.0
  CHARGE min : 0.0


6 sinistres avec CM négatif — des montants faibles (max -5 751€), clairement des recours récupérés ou des régularisations. On les ramène à 0 : ce ne sont pas des sinistres à modéliser.

### 6.2. Cohérence FREQ / CM / CHARGE après correction

On vérifie qu'il n'y a pas d'incohérence résiduelle : un contrat avec FREQ > 0 mais CM = 0 et CHARGE = 0 est un sinistre "fantôme" — il faudra décider si on le compte comme sinistré ou non dans la modélisation.

In [25]:
# Cas incohérents après correction
freq_pos_cm_zero = ((train['FREQ'] > 0) & (train['CM'] == 0)).sum()
freq_pos_charge_neg = ((train['FREQ'] > 0) & (train['CHARGE'] <= 0)).sum()
freq_zero_cm_pos = ((train['FREQ'] == 0) & (train['CM'] > 0)).sum()

print(f"Incohérences après correction :")
print(f"  FREQ > 0 et CM = 0       : {freq_pos_cm_zero}")
print(f"  FREQ > 0 et CHARGE <= 0  : {freq_pos_charge_neg}")
print(f"  FREQ = 0 et CM > 0       : {freq_zero_cm_pos}")

if freq_pos_cm_zero > 0:
    ghosts = train[(train['FREQ'] > 0) & (train['CM'] == 0)]
    print(f"\nDétail des 'sinistres fantômes' (FREQ > 0, CM = 0) :")
    display(ghosts[['ID', 'ANNEE_ASSURANCE', 'FREQ', 'CM', 'CHARGE']].head(10))
    print(f"\n  → Ce sont les {freq_pos_cm_zero} CM négatifs ramenés à 0.")
    print(f"  → Pour la modélisation fréquence : on les comptera comme NON sinistrés (CM = 0 = pas de sinistre)")
    print(f"  → CHARGE déjà à 0 : cohérent")

Incohérences après correction :
  FREQ > 0 et CM = 0       : 542
  FREQ > 0 et CHARGE <= 0  : 542
  FREQ = 0 et CM > 0       : 0

Détail des 'sinistres fantômes' (FREQ > 0, CM = 0) :


,ID,ANNEE_ASSURANCE,FREQ,CM,CHARGE
665,666,1.0000,1.0000,0.0000,0.0000
1150,1151,0.9151,1.0928,0.0000,0.0000
1317,1318,1.0000,1.0000,0.0000,0.0000
1525,1526,0.5452,1.8342,0.0000,0.0000
1744,1745,1.0000,1.0000,0.0000,0.0000
1826,1827,1.0000,1.0000,0.0000,0.0000
2680,2681,1.0000,1.0000,0.0000,0.0000
2829,2830,1.0000,1.0000,0.0000,0.0000
3430,3431,1.0000,1.0000,0.0000,0.0000
3496,3497,0.7479,1.3370,0.0000,0.0000



  → Ce sont les 542 CM négatifs ramenés à 0.
  → Pour la modélisation fréquence : on les comptera comme NON sinistrés (CM = 0 = pas de sinistre)
  → CHARGE déjà à 0 : cohérent


542 contrats ont FREQ > 0 mais CM = 0 et CHARGE = 0. Ce n'est pas seulement les 6 CM négatifs corrigés — il y avait déjà **536 contrats** avec un sinistre déclaré (FREQ > 0) mais aucun coût associé (CM = 0). Ce sont probablement :
- Des sinistres **sans suite** (déclaration mais pas d'indemnisation)
- Des sinistres **en cours de gestion** (pas encore chiffrés au moment de l'extraction)
- Des sinistres **nuls** après expertise (dégâts inférieurs à la franchise)

C'est un point important pour la modélisation :
- **Modèle de fréquence** : ces 542 contrats posent question. Si on définit HAS_SINISTRE = (FREQ > 0), ils sont comptés comme sinistrés. Si on définit HAS_SINISTRE = (CHARGE > 0), ils ne le sont pas. La deuxième définition est plus cohérente avec l'objectif (prédire la charge), mais la première reflète mieux la réalité opérationnelle (un sinistre a bien eu lieu).
- **Modèle de sévérité** : ils seront exclus naturellement (on modélise CM uniquement quand CM > 0).
- **Reconstitution de la charge** : FREQ × CM = quelque chose × 0 = 0. Pas d'impact.

**Décision** : on conserve ces contrats tels quels. Pour le modèle de fréquence, on testera les deux définitions (FREQ > 0 vs CHARGE > 0) et on retiendra celle qui donne le meilleur RMSE sur la charge finale.

### 6.3. Flag plafonnés

In [26]:
plafond = 500000
seuil_plafond = plafond * 0.99

train['IS_PLAFONNE'] = (train['CM'] >= seuil_plafond).astype(int)

n_plaf = train['IS_PLAFONNE'].sum()
charge_plaf = train[train['IS_PLAFONNE'] == 1]['CHARGE'].sum()
charge_tot = train[train['CM'] > 0]['CHARGE'].sum()

print(f"Sinistres plafonnés (CM >= {seuil_plafond:,.0f}€) : {n_plaf}")
print(f"Charge portée : {charge_plaf:,.0f}€ ({charge_plaf/charge_tot*100:.1f}% de la charge sinistres)")

Sinistres plafonnés (CM >= 495,000€) : 20
Charge portée : 10,000,000€ (14.0% de la charge sinistres)


20 sinistres plafonnés, 14% de la charge totale des sinistrés. Le flag `IS_PLAFONNE` permettra de :
- Les **exclure** de l'ajustement GPD (théorie des valeurs extrêmes) — sinon la masse ponctuelle à 500k fausse l'estimation des paramètres de queue
- Les **identifier** dans le modèle de sévérité — ce sont des sinistres censurés, pas des sinistres à 500k pile
- Les **réintégrer** ensuite dans le calcul de la charge totale avec un traitement adapté (extrapolation via la GPD ajustée, ou cap fixe)

<a id="sec-7-recapitulatif-et-sauvegarde"></a>

## 7. Récapitulatif et sauvegarde

Résumé de toutes les transformations appliquées, puis export des datasets prêts pour la modélisation.

### 7.1. Récapitulatif

### Données initiales
- Train input : (383 610, 374)
- Train output : (383 610, 5)
- Test input : (95 852, 374)

### Données finales
- Train : (383 610, 198)
- Test : (95 852, 195)

### Étapes réalisées

| Étape | Détail |
|-------|--------|
| **1. Flags NA** | 4 flags par bloc (météo, socio-démo, RISK, HAUTEUR) + 11 flags individuels = **15 flags**. Vérification : METEO et HAUTEUR quasi-identiques (r=0.90), SOCIO et RISK indépendants → 3 blocs réellement distincts |
| **2. Suppression NA** | 183 variables supprimées (>50% de NA) |
| **3. Imputation** | 31 numériques (médiane), 59 catégorielles (mode), 5 catégorielles (MISSING). Valeurs calculées sur le train uniquement |
| **4. Encodage** | 52 ordinales → numéro d'ordre (tranches régulières vérifiées), 24 binaires N/O → 0/1, 8 trinaires N/R/O → 0/1/2, 6 one-hot (drop_first=False — à ajuster pour les GLM), 3 numériques en texte, 5 ordinales manuelles (TAILLE, COEFASS, AN_EXERC, CARACT4 — mappings vérifiés) |
| **5. Nettoyage** | 2 constantes + 44 quasi-constantes + 24 doublons (corr > 0.98) supprimés. Doublons vérifiés : 10 vrais doublons identiques, 3 quasi-identiques, RISK1/2/3 conservées malgré r=1.0 (structurellement différentes) |
| **6. Feature engineering** | 16 nouvelles variables : 4 ratios, 2 interactions, 5 flags binaires, 2 agrégations (KAPITAL_TOTAL corrigé : 19 montants, exclu 7 ordinaux ; SURFACE_TOTAL : 16 surfaces, aucun ordinal), 3 log-transformations |
| **7. Censure** | 6 CM négatifs ramenés à 0, flag IS_PLAFONNE sur 20 sinistres censurés à 500k. 542 contrats avec FREQ > 0 mais CM = 0 identifiés (sinistres sans suite) |

### Vérifications
- ✅ Zéro NA sur train et test
- ✅ Zéro variable catégorielle restante
- ✅ Alignement train/test parfait (0 colonnes ajoutées/supprimées)
- ✅ Tranches ordinales régulières (vérifiées sur PROPORTION et encodages manuels)
- ✅ Doublons parfaits confirmés avant suppression
- ✅ Agrégations nettoyées (ordinaux exclus de KAPITAL_TOTAL)

### Prochaines étapes
1. Feature engineering avancé (target encoding, interactions, PCA)
2. Estimation du seuil de grave (théorie des valeurs extrêmes)
3. Modélisation (fréquence, CM attritionnels, CM graves)

In [27]:
target_cols = ['FREQ', 'CM', 'CHARGE', 'IS_PLAFONNE']
feature_cols = [c for c in train.columns if c not in ['ID'] + target_cols]

print(f"Features         : {len(feature_cols)}")
print(f"NA train         : {train[feature_cols].isnull().sum().sum()}")
print(f"NA test          : {test_input[[c for c in feature_cols if c in test_input.columns]].isnull().sum().sum()}")
print(f"Catégorielles    : {len(train[feature_cols].select_dtypes(include='object').columns)}")
print(f"Alignement       : {'OK' if set(feature_cols) <= set(test_input.columns) else 'À VÉRIFIER'}")

Features         : 194
NA train         : 0
NA test          : 0
Catégorielles    : 0
Alignement       : OK


### 7.2. Sauvegarde

In [30]:
output_path = "new_data/"

train.to_csv(output_path + "train_engineered.csv", index=False)
test_input.to_csv(output_path + "test_engineered.csv", index=False)

print(f"Train sauvegardé : {train.shape}")
print(f"Test sauvegardé  : {test_input.shape}")
print(f"\nChemin : {output_path}")

Train sauvegardé : (383610, 199)
Test sauvegardé  : (95852, 195)

Chemin : new_data/
